# WetlandMapper: from an outline to wetland maps, statistics and GeoTIFFs

Wetlands Indaba 2026, 90-minute workshop. Conveners: Manudeo Singh and Stephen Tooth (Aberystwyth University) and Ruth-Mary Fisher (SANParks). Runs in Google Colab or on your own computer. You do not need to write code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manudeo/Wetlands_Indaba_2026_workshop/blob/main/notebooks/WetlandMapper_workshop.ipynb)

Everything here is done with [WetlandMapper](https://github.com/manudeo/wetlandmapper) itself: it downloads the satellite record from Google Earth Engine (`fetch`), classifies each pixel's **wetness history** (`classify_dynamics`) and its **Wetland Cover Type** (`classify_wct_ema`), and summarises and plots the results (`summarize_*`, `class_area_timeseries`, `plot_*`). The notebook adds only a few lines for reading your outline, keeping downloads and writing files.

**The workshop plan (90 minutes).** The slides on the science (20 minutes) and a tour of WetlandMapper on GitHub, with its example notebooks and how to report an issue (20 minutes), come first. This notebook is used from minute 40.

| Time | Segment | What you do |
|---|---|---|
| 0:40 | Set up Colab (10 min) | Open this notebook and run the setup cell. Sign in to Earth Engine if you have an account |
| 0:50 | The preset notebook, without Earth Engine (10 to 15 min) | Run Parts A, B and C on the sample data that comes with the notebook |
| about 1:05 | The same areas, with Earth Engine (10 to 15 min) | Run the parts again with Earth Engine on: choose your own months and see a live map |
| about 1:15 | Your own region (the rest of the time) | Load your outline and run the parts on it |

| Part | What it does |
|---|---|
| **A** | **Find the wetlands** in an area (the Agulhas Plain) and compare them with the national wetland map |
| **B** | **One wetland through 36 years**: how its wetness, its cover and the rainfall changed (a pan on the Agulhas Plain) |
| **C** | **Cover types this week** at the Langebaan Ramsar site, to compare with your field visit |

**How to use it.** Press the play button on each cell, in order. Cells with settings show a form on the right; click *Show code* to see what runs. The three examples use sample data that were downloaded in advance and come with the notebook: no account needed. **Your own area and the live "this week" map need Google Earth Engine.**

## Setup

In [ ]:
#@title 0. Setup: run this first (about a minute) { display-mode: "form" }
USE_EARTH_ENGINE = False  #@param {type:"boolean"}
GEE_PROJECT = ""  #@param {type:"string"}
# Tick USE_EARTH_ENGINE and enter your Google Cloud project ID if you have an Earth Engine account.
# You need it for your own area. The three examples work without it.

import os, sys, subprocess, importlib.util, json, re, zipfile, tempfile, shutil, datetime as dt
import xml.etree.ElementTree as ET
from pathlib import Path

REPO_URL = "https://github.com/manudeo/Wetlands_Indaba_2026_workshop.git"
WETLANDMAPPER_VERSION = "1.2.0"
try:
    import google.colab                      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_repo_root():
    # The folder that holds notebooks/ and data/boundaries. Works when the notebook is opened from notebooks/, from the
    # repository root, or from a folder that contains the repository.
    here = Path.cwd().resolve()
    for base in [here, *list(here.parents)[:5]]:
        if (base / "notebooks").is_dir() and (base / "data" / "boundaries").is_dir():
            return base
    for sub in sorted(p for p in here.iterdir() if p.is_dir()):
        if (sub / "notebooks").is_dir() and (sub / "data" / "boundaries").is_dir():
            return sub
    return None

ROOT = find_repo_root()
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from inside the cloned repository (the folder that holds notebooks/ and data/).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)       # a fresh Colab session
    ROOT = Path.cwd() / "Wetlands_Indaba_2026_workshop"
os.chdir(ROOT)

def have(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return False

# Packages: pinned and installed in Colab; on your own computer they are checked, never changed.
extra = "[gee]" if USE_EARTH_ENGINE else ""
MODULES = {"wetlandmapper": f"wetlandmapper{extra}=={WETLANDMAPPER_VERSION}", "geopandas": "geopandas", "rioxarray": "rioxarray", "netCDF4": "netCDF4", "scipy": "scipy"}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *MODULES.values()], check=True)
else:
    missing = [pip for mod, pip in MODULES.items() if not have(mod)]
    if missing:
        raise ImportError("Missing packages: " + ", ".join(missing) + ".\nInstall them in your environment, for example:\n  pip install " + " ".join(f'"{m}"' for m in missing))

import warnings
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd, rioxarray, shapely
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from scipy import ndimage
from scipy.stats import spearmanr
from shapely.geometry import box, shape as shp_shape

import wetlandmapper as wm
from wetlandmapper import (classify_dynamics, classify_wct_ema, compute_wet_frequency, detect_wet_events, build_run_manifest,
                           DYNAMICS_CLASSES, DYNAMICS_COLORS, WCT_CLASSES, WCT_COLORS)
from wetlandmapper.analysis import summarize_dynamics, summarize_wct, class_area_timeseries, summarize_by_polygons, linear_trend
from wetlandmapper.plotting import plot_dynamics, plot_wct

warnings.filterwarnings("ignore", category=FutureWarning); warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11, "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False})
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
CACHE_IN_DRIVE = False  #@param {type:"boolean"}
# On your own computer downloads are kept in data/cache inside the repository and stay there. In Colab that folder is deleted with the session;
# tick CACHE_IN_DRIVE to keep downloads in My Drive/WetlandMapper_cache instead (the repository's sample files are copied there the first time).
CACHE = ROOT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
if CACHE_IN_DRIVE:
    if not IN_COLAB:
        print("CACHE_IN_DRIVE is for Colab; on your own computer the cache is", CACHE.resolve())
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_cache = Path("/content/drive/MyDrive/WetlandMapper_cache"); drive_cache.mkdir(parents=True, exist_ok=True)
        for f in CACHE.glob("*"):
            if f.is_file() and not (drive_cache / f.name).exists():
                shutil.copy(f, drive_cache / f.name)
        CACHE = drive_cache
print("Downloads are kept in", CACHE)

GEE_READY = False
if USE_EARTH_ENGINE:
    try:
        from wetlandmapper import gee
        from wetlandmapper.gee import fetch
        try:
            gee.init(project=GEE_PROJECT)            # reuse saved credentials (your own computer, or an earlier sign-in)
        except Exception:
            gee.authenticate(); gee.init(project=GEE_PROJECT)
        GEE_READY = True
        print("Earth Engine ready.")
    except Exception as e:
        print(f"Earth Engine sign-in failed ({type(e).__name__}: {str(e)[:150]}). Check the project ID and that the Earth Engine API is enabled for it.")
else:
    print("Earth Engine is off: the three examples run on cached sample data. Your own area and the live map need Earth Engine.")

def need_gee(what):
    if not GEE_READY:
        raise RuntimeError(f"{what} needs Earth Engine. Tick USE_EARTH_ENGINE in the setup cell, enter your project ID and run it again.")

# ---------------------------------------------------------------- outlines and points
def read_kml_polygons(path):
    # Fallback KML/KMZ reader for when GDAL has no KML driver.
    p = Path(path)
    data = zipfile.ZipFile(p).read(next(n for n in zipfile.ZipFile(p).namelist() if n.lower().endswith(".kml"))) if p.suffix.lower() == ".kmz" else p.read_bytes()
    tag = lambda e: e.tag.split("}")[-1]
    ring = lambda b: [tuple(float(v) for v in t.split(",")[:2]) for t in next(c for c in b.iter() if tag(c) == "coordinates").text.split()]
    polys = []
    for poly in ET.fromstring(data).iter():
        if tag(poly) == "Polygon":
            outer, holes = None, []
            for b in poly:
                if tag(b) == "outerBoundaryIs": outer = ring(b)
                elif tag(b) == "innerBoundaryIs": holes.append(ring(b))
            if outer and len(outer) >= 4: polys.append(shapely.Polygon(outer, holes))
    return gpd.GeoDataFrame({"geometry": polys}, geometry="geometry", crs=4326)

def read_outline(path):
    # Polygons from .kmz, .kml, .gpkg, .geojson, a zipped shapefile or a .shp, as 2-D geometries in EPSG:4326.
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{p} not found.")
    try:
        g = gpd.read_file(f"zip://{p}") if p.suffix.lower() == ".zip" else gpd.read_file(p)
        ok = g.geom_type.isin(["Polygon", "MultiPolygon"]).any()
    except Exception:
        g, ok = None, False
    if not ok and p.suffix.lower() in (".kml", ".kmz"):
        g = read_kml_polygons(p)
    if g is None or g.crs is None:
        raise ValueError(f"{p.name} has no readable polygons or no coordinate system (a shapefile needs its .prj; or save as GeoPackage).")
    g = g.to_crs(4326)
    g = g.set_geometry(shapely.force_2d(g.geometry.values), crs=4326)
    g = g[g.geom_type.isin(["Polygon", "MultiPolygon"])]
    if g.empty:
        raise ValueError(f"No polygons found in {p.name}: points and lines are not enough. Draw the area as a polygon (Google Earth: Add, Polygon) and save again.")
    return g.set_geometry([shapely.make_valid(x) for x in g.geometry], crs=4326).reset_index(drop=True)

def read_points(path):
    # Field points from a CSV with lat and lon columns, or from a kmz, kml, gpkg, shp or geojson of points.
    p = Path(path)
    if p.suffix.lower() in (".csv", ".txt"):
        df = pd.read_csv(p); low = {c.lower().strip(): c for c in df.columns}
        la = next((low[k] for k in ("lat", "latitude", "y") if k in low), None); lo = next((low[k] for k in ("lon", "lng", "long", "longitude", "x") if k in low), None)
        if la is None or lo is None:
            raise ValueError("The table needs latitude (lat) and longitude (lon) columns in decimal degrees.")
        g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lo], df[la]), crs=4326)
    else:
        g = gpd.read_file(p).to_crs(4326); g = g.set_geometry(shapely.force_2d(g.geometry.values), crs=4326); g = g[g.geom_type == "Point"]
    g = g.reset_index(drop=True); g.insert(0, "point_id", [f"P{i + 1}" for i in range(len(g))])
    return g

def upload_file():
    # Colab only: open the upload dialog and return the path of the uploaded file (a shapefile also needs its .dbf, .shx and .prj).
    try:
        from google.colab import files
    except ImportError as e:
        raise RuntimeError("Uploading works in Google Colab only. On your own computer choose 'path' and give the full path of the file.") from e
    up = files.upload(); tmp = Path(tempfile.mkdtemp(prefix="wm_upload_"))
    for n, b in up.items(): (tmp / Path(n).name).write_bytes(b)
    order = [".gpkg", ".kmz", ".kml", ".geojson", ".json", ".zip", ".shp", ".csv"]
    found = sorted((q for q in tmp.iterdir() if q.suffix.lower() in order), key=lambda q: order.index(q.suffix.lower()))
    if not found: raise ValueError("No readable file uploaded. Supported: " + ", ".join(order))
    return found[0]

def load_outline(source, path, example_path):
    # The outline chosen in a form: upload, a path, the example file, or none.
    if source == "none":
        return None
    return read_outline(upload_file() if source == "upload" else (path if source == "path in Drive or Colab" else example_path))

def describe_outline(gdf, name, wetland):
    # Size, a plain note on whether it suits the part it is for, and a small map.
    c = shapely.unary_union(gdf.geometry.values).centroid
    ha = gdf.to_crs((32700 if c.y < 0 else 32600) + int((c.x + 180) // 6) + 1).area.sum() / 1e4
    w, s_, e, n = gdf.total_bounds; step = 30 / 111319.49; nx, ny = int((e - w) / step), int((n - s_) / step)
    print(f"Outline '{name}': {len(gdf)} polygon(s), {ha:,.1f} ha, box {nx} x {ny} pixels at 30 m")
    if nx * ny > 2.5e6:
        raise ValueError("The outline's box is too large for this notebook (over about 2.5 million pixels per year). Use a smaller outline.")
    if wetland and nx * ny > 4e5:
        print("Note: this is a large outline for one wetland. Parts B and C report on it as a whole. To search a larger area for wetlands, use the area cell and Part A.")
    if not wetland and ha < 100:
        print("Note: this is a small area for finding wetlands. To report on one wetland, use the wetland cell and Parts B and C.")
    fig, ax = plt.subplots(figsize=(4.5, 4)); gdf.plot(ax=ax, color="#76d7c4", edgecolor="k"); ax.set_title(name); ax.set_aspect(1 / np.cos(np.radians(float(c.y)))); plt.show()

def geojson_of(gdf):
    return shapely.unary_union(gdf.geometry.values).__geo_interface__

def bounds_of(gdf, pad=0.004):
    w, s, e, n = gdf.total_bounds
    return (round(w - pad, 5), round(s - pad, 5), round(e + pad, 5), round(n + pad, 5))

# ---------------------------------------------------------------- the example areas
BOUND = ROOT / "data" / "boundaries"
def example_area(name):
    if name == "agulhas":                                    # the box around the national wetland map polygons of the Agulhas Plain
        g = gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp").to_crs(4326)
        return gpd.GeoDataFrame(geometry=[box(*bounds_of(g, 0.005))], crs=4326)
    if name == "pan":                                        # the study window: the bounding box of pan OBJECTID 3462 with a 1 km buffer
        g = gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp"); g = g[g["OBJECTID"] == 3462]
        return gpd.GeoDataFrame(geometry=[box(*bounds_of(gpd.GeoDataFrame(geometry=g.to_crs(32734).buffer(1000).to_crs(4326).values, crs=4326), 0))], crs=4326)
    if name == "pan_polygon":
        g = gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp"); return g[g["OBJECTID"] == 3462].to_crs(4326).reset_index(drop=True)
    if name == "langebaan":
        return gpd.read_file(BOUND / "Langebaan_Ramsar.shp").to_crs(4326)

def langebaan_box():
    # The request box for Langebaan: the polygon's bounding box plus a little, as in earlier downloads (the data are then masked to the polygon).
    return gpd.GeoDataFrame(geometry=[box(*bounds_of(example_area("langebaan"), 0.003))], crs=4326)

def mask_to(obj, gdf):
    return obj.rio.clip(gdf.geometry, gdf.crs, drop=False, all_touched=True)

# ---------------------------------------------------------------- downloads with WetlandMapper's fetch, and a cache
def _years(obj):
    first = obj if isinstance(obj, xr.DataArray) else obj[list(obj.data_vars)[0]]
    return set(int(y) for y in pd.DatetimeIndex(first["time"].values).year)

def fetch_annual(geom, index, y0, y1, **kw):
    # WetlandMapper's fetch for one composite per year; years skipped by a failed download are fetched again.
    need_gee("Downloading satellite data")
    base = dict(aoi=geom, sensor="LandsatAll", scale=30, temporal_aggregation="annual", use_slc_off=True, project=GEE_PROJECT or None)
    out = fetch(start=f"{y0}-01-01", end=f"{y1}-12-31", index=index, **base, **kw)
    for attempt in range(2):
        miss = sorted(set(range(y0, y1 + 1)) - _years(out))
        if not miss: break
        print(f"  fetching {len(miss)} missing year(s) again: {miss}")
        parts = []
        for y in miss:
            try: parts.append(fetch(start=f"{y}-01-01", end=f"{y}-12-31", index=index, **base, **kw))
            except Exception as e: print(f"    {y}: {type(e).__name__}")
        parts = [q for q in parts if len(_years(q))]
        if parts: out = xr.concat([out] + parts, dim="time").sortby("time")
    return out

def fin(da):
    return da.astype("float32").rio.write_crs("EPSG:4326")

def annual_aweish(geom, y0, y1, months=None):
    # Annual AWEIsh composite: the 90th percentile of the cloud-free observations of each year, over all months or over the months given
    # (water is seasonal in dry places, so the choice of months matters).
    kw = dict(months=list(months)) if months else {}
    da = fetch_annual(geom, "AWEIsh", y0, y1, reduction_method="percentile", percentile=90.0, **kw)
    return fin(da["AWEIsh"] if isinstance(da, xr.Dataset) else da)

MONTH_NAMES = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

def choose_months(mode, first_month, last_month, suggested=None):
    # Months from the drop-downs in a form. "all year" gives None. "suggested rainy season" gives the suggestion from the rainfall step.
    # "my own range of months" runs from the first month to the last month, both included; if the first month is later than the last, the range crosses New Year.
    if mode == "all year":
        return None
    if mode == "suggested rainy season":
        if not suggested:
            raise ValueError("The suggested rainy season comes from the rainfall step (Part B, step 1). Run it first, or choose your own range of months.")
        return list(suggested)
    a, b = MONTH_NAMES.index(first_month) + 1, MONTH_NAMES.index(last_month) + 1
    return list(range(a, b + 1)) if a <= b else list(range(a, 13)) + list(range(1, b + 1))

def check_dynamics_settings(n_years, years_per_period, wet_pct, persistent_pct):
    # The record is split into an early period (its first N years) and a recent period (its last N years).
    if 2 * years_per_period > n_years:
        raise ValueError(f"The record has {n_years} years, so each period can have at most {n_years // 2} years (you chose {years_per_period}). Lower 'years in each period'.")
    if persistent_pct < wet_pct:
        raise ValueError("'Persistent' needs a higher percentage than 'wet in at least': raise the persistent percentage or lower the other.")
    print(f"Settings: a pixel is wetland if it is wet in at least {wet_pct}% of years, and persistent if wet in at least {persistent_pct}%. "
          f"Early period = the first {years_per_period} years, recent period = the last {years_per_period} years.")

def class_table_for_display(df):
    # The class summary with plain column names.
    return df[["class", "pixel_count", "area_ha", "percent_of_valid"]].rename(columns={"class": "Class", "pixel_count": "Pixels", "area_ha": "Area (ha)", "percent_of_valid": "% of the area analysed"}).round({"Area (ha)": 1, "% of the area analysed": 1})

def wet_season_indices(geom, y0, y1, months):
    # MNDWI, NDVI and NDTI: the median of the rainy-season months of each year.
    ds = fetch_annual(geom, ["MNDWI", "NDVI", "NDTI"], y0, y1, months=list(months), reduction_method="median")
    return xr.Dataset({v: fin(ds[v]) for v in ("MNDWI", "NDVI", "NDTI")})

def save_nc(obj, path):
    # Scaled int16 netCDF, written to a temporary file first so a failed write never leaves a broken file.
    ds = obj.to_dataset() if isinstance(obj, xr.DataArray) else obj
    ds = ds.rio.write_crs("EPSG:4326")
    for v in ds.data_vars:
        for k in ("_FillValue", "missing_value", "scale_factor", "add_offset"): ds[v].attrs.pop(k, None)
        ds[v].encoding.clear()
    enc = {v: {"dtype": "int16", "scale_factor": 0.001, "add_offset": 0.0, "_FillValue": -32768, "zlib": True, "complevel": 5} for v in ds.data_vars}
    tmp = Path(path).with_name(Path(path).stem + ".partial.nc")
    try:
        ds.to_netcdf(tmp, encoding=enc); os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)

def cached(name, make, force=False):
    # Read data/cache/<name>.nc if it exists; otherwise download with make() (needs Earth Engine) and keep a copy.
    f = CACHE / f"{name}.nc"
    if f.exists() and not force:
        ds = xr.open_dataset(f)
        if len(ds.data_vars) == 0:
            ds.close(); raise FileNotFoundError(f"{f.name} is empty (an interrupted download). Delete it and run the cell again with Earth Engine on.")
        ds = ds.load()
        return fin(ds[list(ds.data_vars)[0]]) if len(ds.data_vars) == 1 else xr.Dataset({v: fin(ds[v]) for v in ds.data_vars})
    if not GEE_READY:
        raise FileNotFoundError(f"The sample data '{name}' are not in data/cache/ yet. Ask the workshop leader, or switch on Earth Engine to download them.")
    print(f"Downloading '{name}' from Earth Engine (kept in data/cache/ for next time)...")
    out = make(); save_nc(out, f); return out

def fill_gaps(name, make_year, polygon=None, y0=1990, y1=2025):
    # Download again only the years that are missing, or hold no valid data, in a cached file (for example after switching on Landsat 7
    # after 2003), and keep every other year as it is.
    f = CACHE / f"{name}.nc"
    if not f.exists():
        return
    old = cached(name, None)
    first = old if isinstance(old, xr.DataArray) else old[list(old.data_vars)[0]]
    inside = inside_mask(polygon, first) if polygon is not None else xr.ones_like(first.isel(time=0, drop=True), dtype=bool)
    yrs = np.asarray(pd.DatetimeIndex(first.time.values).year)
    has = (first.notnull() & inside).any(("y", "x")).values
    bad = sorted(set(range(y0, y1 + 1)) - set(int(y) for y in yrs[has]))
    if not bad:
        print(f"{name}: no gaps"); return
    print(f"{name}: downloading {len(bad)} year(s) again: {bad}")
    fixed = []
    for y in bad:
        try:
            new = make_year(y)
            if polygon is not None: new = mask_to(new, polygon)
            new = new.reindex(y=first.y, x=first.x, method="nearest", tolerance=1e-4)
            ok = new[list(new.data_vars)[0]] if isinstance(new, xr.Dataset) else new
            if bool((ok.notnull() & inside).any()): fixed.append((y, new))
            else: print(f"  {y}: still no usable scenes")
        except Exception as e:
            print(f"  {y}: {type(e).__name__}: {str(e)[:80]}")
    if fixed:
        keep = ~np.isin(yrs, [y for y, _ in fixed])
        merged = xr.concat([old.isel(time=np.where(keep)[0])] + [n for _, n in fixed], dim="time").sortby("time")
        save_nc(merged, f); print(f"  {name}: filled {[y for y, _ in fixed]}")

def rain_monthly(bounds, y0, y1):
    # Monthly rainfall (mm) at the area from ERA5-Land through Earth Engine.
    need_gee("Rainfall")
    import ee
    geom = ee.Geometry.Rectangle(list(bounds))
    col = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01").select("total_precipitation_sum")
    def f(img):
        d = img.reduceRegion(ee.Reducer.mean(), geom, 11132, maxPixels=int(1e8))
        return ee.Feature(None, {"date": img.date().format("YYYY-MM"), "p": d.get("total_precipitation_sum")})
    feats = ee.FeatureCollection(col.map(f)).getInfo()["features"]
    df = pd.DataFrame([x["properties"] for x in feats]); df["precip_mm"] = df["p"].astype(float) * 1000.0; df["date"] = pd.to_datetime(df["date"] + "-01")
    return df[["date", "precip_mm"]].sort_values("date").reset_index(drop=True)

def cached_rain(name, bounds, y0, y1, force=False):
    f = CACHE / f"{name}.csv"
    if f.exists() and not force: return pd.read_csv(f, parse_dates=["date"])
    if not GEE_READY: raise FileNotFoundError(f"The rainfall table '{name}' is not in data/cache/ yet. Ask the workshop leader, or switch on Earth Engine.")
    df = rain_monthly(bounds, y0, y1); df.to_csv(f, index=False); return df

def suggest_months(rain, width=5):
    # The wettest run of months inside one calendar year (yearly composites are built by calendar year).
    clim = rain.assign(m=rain["date"].dt.month).groupby("m")["precip_mm"].mean().reindex(range(1, 13)).fillna(0)
    best = max(range(1, 13 - width + 1), key=lambda m0: clim.loc[m0:m0 + width - 1].sum())
    wrap = max(range(1, 13), key=lambda m0: sum(clim.loc[((m0 - 1 + k) % 12) + 1] for k in range(width)))
    return list(range(best, best + width)), [((wrap - 1 + k) % 12) + 1 for k in range(width)], clim

# ---------------------------------------------------------------- small numerical helpers
def pixel_area_m2(da):
    dx, dy = (abs(float(r)) for r in da.rio.resolution())
    return dx * 111320 * np.cos(np.radians(float(da.y.mean()))) * dy * 110574

def inside_mask(gdf, like):
    ref = like.isel(time=0, drop=True) if "time" in like.dims else like
    def r(t): return rasterize([(g, 1) for g in gdf.geometry], out_shape=ref.shape, transform=ref.rio.transform(), fill=0, dtype="uint8", all_touched=t).astype(bool)
    m = r(False)
    return xr.DataArray(m if m.any() else r(True), dims=ref.dims, coords=ref.coords)

def utm_epsg(da):
    return (32700 if float(da.y.mean()) < 0 else 32600) + int((float(da.x.mean()) + 180) // 6) + 1

def write_geotiff(da, path, colors):
    # GeoTIFF in the local UTM zone (uint8, 255 = no data) with the class colours embedded, so QGIS shows them without a style file.
    r = max(round(abs(float(da.rio.resolution()[0])) * 111320 * np.cos(np.radians(float(da.y.mean()))) / 5) * 5, 5)
    u = da.rio.write_crs("EPSG:4326").rio.reproject(f"EPSG:{utm_epsg(da)}", resolution=r, resampling=Resampling.nearest)
    arr = u.fillna(255).astype("uint8").values
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1, dtype="uint8", crs=u.rio.crs, transform=u.rio.transform(), nodata=255, compress="deflate") as dst:
        dst.write(arr, 1)
        dst.write_colormap(1, {**{int(c): tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)) + (255,) for c, h in colors.items()}, 255: (0, 0, 0, 0)})

def wetland_patches(dyn, wf, min_pixels=2):
    # Groups of touching wetland pixels (class above 0) as shapes in UTM, with area, main class and the share of years they were wet.
    epsg = utm_epsg(dyn); res = max(round(abs(float(dyn.rio.resolution()[0])) * 111320 * np.cos(np.radians(float(dyn.y.mean()))) / 5) * 5, 5)
    utm = xr.Dataset({"cls": dyn.astype("float32"), "wf": wf.astype("float32")}).rio.write_crs("EPSG:4326").rio.reproject(f"EPSG:{epsg}", resolution=res, resampling=Resampling.nearest)
    cls = utm["cls"].values; lab, n = ndimage.label(np.isfinite(cls) & (cls > 0), structure=np.ones((3, 3)))
    empty = gpd.GeoDataFrame({"patch_id": [], "area_ha": []}, geometry=[], crs=epsg)
    if n == 0: return empty
    sizes = ndimage.sum(lab > 0, lab, np.arange(1, n + 1)); keep = np.where(sizes >= min_pixels)[0] + 1
    remap = np.zeros(n + 1, dtype=np.int32); remap[keep] = np.arange(1, len(keep) + 1); lab = remap[lab]; ids = np.arange(1, len(keep) + 1)
    if len(ids) == 0: return empty
    dom = pd.DataFrame({"o": lab[lab > 0], "c": cls[lab > 0].astype(int)}).groupby("o")["c"].agg(lambda s: s.value_counts().idxmax())
    geoms = [(shp_shape(g), int(v)) for g, v in shapes(lab.astype("int32"), mask=lab > 0, transform=utm.rio.transform(), connectivity=8)]
    gdf = gpd.GeoDataFrame({"patch_id": [v for _, v in geoms]}, geometry=[g for g, _ in geoms], crs=epsg).dissolve(by="patch_id").reset_index()
    gdf["area_ha"] = (ndimage.sum(lab > 0, lab, gdf["patch_id"]) * res * res / 1e4).round(2)
    gdf["main_class"] = gdf["patch_id"].map(dom).map(DYNAMICS_CLASSES); gdf["percent_years_wet"] = ndimage.mean(utm["wf"].values, lab, gdf["patch_id"]).round(1)
    return gdf

def write_kmz(gdf, path):
    # Wetland patches for Google Earth or a phone.
    try:
        g = gdf[["patch_id", "area_ha", "main_class", "geometry"]].to_crs(4326); tmp = Path(tempfile.mkdtemp()) / "doc.kml"; g.to_file(tmp, driver="KML")
        with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z: z.write(tmp, "doc.kml")
    except Exception as e:
        print(f"  (KMZ not written: {type(e).__name__}; the GeoPackage has the same patches)")

def usable_years(ts_ds, mask_px, class_code, frac=0.8):
    # One value per year from class_area_timeseries; years with valid data for under 80% of the area are NaN (no data is not zero).
    d = ts_ds.to_dataframe().reset_index(); d["year"] = pd.to_datetime(d["time"]).dt.year
    tot = d.groupby("year")["total_valid_pixels"].first(); col = [c for c in d.columns if c.startswith("area_")][0]
    s = d[d["class_code"] == class_code].set_index("year")[col]
    return s.where(tot.reindex(s.index) >= frac * mask_px)

def show(df):
    from IPython.display import display
    display(df)

print("WetlandMapper", wm.__version__, "|", "Colab" if IN_COLAB else "local", "| repository:", ROOT)

## Your own outlines (optional, do this when you are ready)

An outline is a **polygon**: KMZ or KML (drawn in Google Earth: Add, Polygon), GeoPackage, GeoJSON, a zipped shapefile, or a shapefile with its `.dbf`, `.shx` and `.prj`. Points and lines are not enough.

You can load **two different outlines**, because the parts ask different questions:

| Outline | Used by | What it is |
|---|---|---|
| **Area to search** | Part A | A larger area, such as a catchment, municipality, reserve or farm. The notebook finds the wetlands inside it |
| **Your wetland** | Parts B and C | One wetland, or several, each as its own polygon. The notebook reports on the wetland itself: wet area through time, rainfall, cover types, and a recent map |

Run only the cell you need. Each prints the size of your outline and a note if it looks like the wrong kind. In the part, set *AREA* to **my own** area or wetland. If you have no outline of your own, each cell has an example file to test the route: a 7 km area around the pan, and the pan itself.

In [ ]:
#@title Your own area to search (Part A) { display-mode: "form" }
REGION_NAME = "my_area"  #@param {type:"string"}
REGION_OUTLINE = "none"  #@param ["none", "upload", "path in Drive or Colab", "example: 7 km area around the pan (KMZ)"]
REGION_PATH = ""  #@param {type:"string"}
own_region = load_outline(REGION_OUTLINE, REGION_PATH, ROOT / "data" / "examples" / "pan_region_outline.kmz")
if own_region is not None:
    describe_outline(own_region, REGION_NAME, wetland=False)

In [ ]:
#@title Your wetland outline (Parts B and C) { display-mode: "form" }
WETLAND_NAME = "my_wetland"  #@param {type:"string"}
WETLAND_OUTLINE = "none"  #@param ["none", "upload", "path in Drive or Colab", "example: pan 3462 (KMZ)"]
WETLAND_PATH = ""  #@param {type:"string"}
own_wetland = load_outline(WETLAND_OUTLINE, WETLAND_PATH, ROOT / "data" / "examples" / "pan3462_outline.kmz")
if own_wetland is not None:
    describe_outline(own_wetland, WETLAND_NAME, wetland=True)

## Part A. Find the wetlands in an area: the Agulhas Plain

**The question.** Which places in this landscape behave like wetland, and how does that compare with the national wetland map?

**How it works.** WetlandMapper looks at one satellite image per year for 36 years (1990 to 2025). In each image every 30 m pixel (0.09 ha) is either wet or dry: it counts as wet when its water index (AWEIsh, which is above zero over water) is above zero. For each year the image keeps the wettest-looking cloud-free value of the pixel (the 90th percentile of that year's observations), because in dry places water often comes only briefly.

A pixel that was wet in enough years counts as **wetland**, and its **wetness history** gives it a class (see *How to read it*). Touching wetland pixels are grouped into **wetland patches**: shapes with an area, a main class and the share of years they were wet. For the example, the result is compared with the wetland polygons of the Agulhas Plain in the National Wetland Map 5 (supplied by SANParks).

To search your own area, load it in the *area to search* cell above and set *AREA* to **my own area**.

**The settings**

- **Wet in at least this % of years** (default 25): how often a pixel must be wet to count as wetland.
- **Persistent if wet in at least this % of years** (default 75): above this, a pixel is *Persistent*.
- **Years per period** (default 3): the record is split into an early period (its first N years) and a recent period (its last N years), and the two are compared. The 2022 paper used 5.
- **Smallest patch**: patches with fewer pixels than this are dropped.
- **Months**: which months of each year go into the yearly image. The default is *all year*. In a place with a clear rainy season you may prefer only those months: the rainfall chart in Part B, step 1 shows which they are. With Earth Engine on, your months are downloaded and used; without it the sample data (all months) are used and the notebook says so.

In [ ]:
#@title Part A. Find the wetlands { display-mode: "form" }
AREA = "example: Agulhas Plain"  #@param ["example: Agulhas Plain", "my own area"]
MINIMUM_WET_YEARS_PERCENT = 25  #@param {type:"slider", min:5, max:60, step:5}
PERSISTENT_YEARS_PERCENT = 75  #@param {type:"slider", min:50, max:100, step:5}
YEARS_PER_PERIOD = 3  #@param {type:"slider", min:1, max:10, step:1}
SMALLEST_PATCH_PIXELS = 2  #@param {type:"slider", min:1, max:10, step:1}
FIRST_YEAR_A = 1990  #@param {type:"integer"}
LAST_YEAR_A = 2025  #@param {type:"integer"}
MONTHS_A = "all year"  #@param ["all year", "my own range of months"]
FIRST_MONTH_A = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
LAST_MONTH_A = "December"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
# MINIMUM_WET_YEARS_PERCENT: a pixel counts as wetland if it was wet in at least this % of the years.
# PERSISTENT_YEARS_PERCENT: a pixel is 'Persistent' if it was wet in at least this % of the years.
# YEARS_PER_PERIOD: the record is split into an early period (its first N years) and a recent period (its last N years), and the two are compared.
#   The 2022 paper used 5 years.
# SMALLEST_PATCH_PIXELS: wetland patches smaller than this are dropped (1 pixel = 0.09 ha).
# FIRST_YEAR_A, LAST_YEAR_A: used for your own area only (the example always uses 1990 to 2025).
# MONTHS_A: which months of each year go into the yearly image. "all year" (the example), or a range: pick the first and last month (both included).
#   To see which months are wet at your site, look at the rainfall chart in Part B, step 1.

example_a = AREA.startswith("example")
months_a = choose_months(MONTHS_A, FIRST_MONTH_A, LAST_MONTH_A)                      # None means all year
if example_a:
    aoi_a, name_a, ref = example_area("agulhas"), "agulhas_plain", gpd.read_file(BOUND / "Ramsar_AgulhasPlain_wetlands.shp").to_crs(4326)
    if months_a is not None and not GEE_READY:
        print(f"Earth Engine is off, so the sample data (all months of each year) are used instead of months {months_a}. Choosing months needs Earth Engine.")
        months_a = None
    print("Data:", "sample data that come with the notebook" if months_a is None else "downloaded from Earth Engine for your months (a few minutes)")
    aw_a = cached("agulhas_aweish_annual", lambda: annual_aweish(geojson_of(aoi_a), 1990, 2025)) if months_a is None else annual_aweish(geojson_of(aoi_a), 1990, 2025, months=months_a)
else:
    assert own_region is not None, "Run the 'Your own area to search' cell first."
    aoi_a, name_a, ref = own_region, REGION_NAME + "_find", None
    aw_a = annual_aweish(geojson_of(aoi_a), FIRST_YEAR_A, LAST_YEAR_A, months=months_a)
out_a = OUT / name_a; out_a.mkdir(exist_ok=True)
print("Yearly image: the 90th percentile of the water index (AWEIsh) over", "all months of each year" if months_a is None else f"months {months_a} of each year")
check_dynamics_settings(aw_a.sizes["time"], YEARS_PER_PERIOD, MINIMUM_WET_YEARS_PERCENT, PERSISTENT_YEARS_PERCENT)

mask_a = inside_mask(aoi_a, aw_a); ok_a = aw_a.notnull().any("time") & mask_a
dyn_a = classify_dynamics(aw_a, nYear=YEARS_PER_PERIOD, thresholdWet=MINIMUM_WET_YEARS_PERCENT, thresholdPersis=PERSISTENT_YEARS_PERCENT, nan_policy="valid").where(ok_a)
wf_a = compute_wet_frequency(aw_a, nan_policy="valid").where(ok_a)
px_a = pixel_area_m2(aw_a)
classes_a = summarize_dynamics(dyn_a, pixel_area=px_a, area_unit="ha").to_dataframe().drop(columns=["class_name"]).join(pd.Series(DYNAMICS_CLASSES, name="class"))
patches_a = wetland_patches(dyn_a, wf_a, SMALLEST_PATCH_PIXELS)
wet_ha = float(classes_a.loc[classes_a.index != 0, "area_ha"].sum()); tot_ha = float(ok_a.sum()) * px_a / 1e4
print(f"Wetland found: {wet_ha:,.0f} ha of the {tot_ha:,.0f} ha analysed ({100 * wet_ha / tot_ha:.1f}%), in {len(patches_a):,} patches of at least {SMALLEST_PATCH_PIXELS} pixels.")
if len(patches_a): print(f"The median patch is {patches_a['area_ha'].median():.2f} ha and the largest is {patches_a['area_ha'].max():,.1f} ha.")
yrs_a = sorted(set(pd.DatetimeIndex(aw_a.time.values).year)); gaps_a = [y for y in range(yrs_a[0], yrs_a[-1] + 1) if y not in yrs_a]
if gaps_a: print(f"No yearly image for {gaps_a}: those years are left out.")
show(class_table_for_display(classes_a))

fig, ax = plot_dynamics(dyn_a, title=f"{name_a}: wetness history {yrs_a[0]} to {yrs_a[-1]}", figsize=(11, 5.4))
if ref is not None: ref.boundary.plot(ax=ax, color="k", linewidth=0.4)
ax.set_aspect(1 / np.cos(np.radians(float(dyn_a.y.mean())))); fig.savefig(out_a / f"{name_a}_wetness_history_map.png", dpi=150, bbox_inches="tight"); plt.show()

if ref is not None:                                           # compare with the national wetland map, by wetland type
    ref = ref.reset_index(drop=True).assign(poly=lambda d: d.index, wetland_type=lambda d: d["Ramsar"].astype(str).str.replace(r"^\d+\s*", "", regex=True))
    detected = (dyn_a > 0).where(dyn_a.notnull()).astype("float32")              # 1 where the satellite record shows wetland
    z = summarize_by_polygons(detected, ref[["poly", "geometry"]], polygon_id_col="poly", stats=("mean", "count")).merge(ref[["poly", "wetland_type"]], left_on="polygon_id", right_on="poly")
    z["detected_px"] = z["mean"] * z["count"]
    agree = z.groupby("wetland_type").agg(map_ha=("count", lambda c: c.sum() * px_a / 1e4), detected_ha=("detected_px", lambda c: c.sum() * px_a / 1e4))
    agree.insert(0, "n_wetlands", ref.groupby("wetland_type").size())          # all wetlands in the map; those smaller than a pixel add no area
    agree["detected_pct"] = (100 * agree["detected_ha"] / agree["map_ha"]).round(0); agree = agree.round(1)
    outside = patches_a[~patches_a.intersects(shapely.unary_union(ref.to_crs(patches_a.crs).geometry.values))].sort_values("area_ha", ascending=False)
    print(f"\nOf the {agree['map_ha'].sum():,.0f} ha in the national wetland map, {agree['detected_ha'].sum():,.0f} ha ({100 * agree['detected_ha'].sum() / agree['map_ha'].sum():.0f}%) also shows as wetland in the satellite record.")
    print(f"The satellite record also shows {len(outside)} wetland patches ({outside['area_ha'].sum():,.0f} ha) that are outside the national map. They are places to check, not new wetlands.\n")
    if example_a:
        print("Note: the largest patches outside the map are along the southern edge of the study box. They are the sea, which shows as persistent wetland and inflates the wetland total above. Open sea, large lakes and reservoirs all behave like this: check the largest patches on a map.\n")
    agree_show = agree.rename(columns={"n_wetlands": "Wetlands in the map", "map_ha": "Area in the map (ha)", "detected_ha": "Of that, also wetland in the satellite record (ha)", "detected_pct": "% of the map area also found by the satellite"})
    print("How well the satellite record agrees with the national wetland map, by wetland type:"); show(agree_show)
    fig, ax = plt.subplots(figsize=(8, 3.4)); t = agree.sort_values("map_ha")
    ax.barh(t.index, t["detected_pct"], color="#1a5276"); ax.set_xlim(0, 118); ax.set_xlabel("% of the mapped area that also shows as wetland in the satellite record")
    ax.set_title("National wetland map, by wetland type: how much does the satellite record agree?", fontsize=10)
    for i, (v, n) in enumerate(zip(t["detected_pct"], t["n_wetlands"])): ax.text(v + 1, i, f"{v:.0f}%   ({n} wetland{'s' if n != 1 else ''} in the map)", va="center", fontsize=8)
    fig.tight_layout(); fig.savefig(out_a / f"{name_a}_agreement_with_national_map.png", dpi=150, bbox_inches="tight"); plt.show()
    agree_show.to_csv(out_a / f"{name_a}_agreement_with_national_map.csv")
    outside.drop(columns="geometry").head(25).to_csv(out_a / f"{name_a}_patches_not_in_national_map.csv", index=False)
    print("The largest satellite wetland patches that are not in the national map (places to check):")
    show(outside.drop(columns="geometry").head(8).set_index("patch_id").rename(columns={"area_ha": "Area (ha)", "main_class": "Main class", "percent_years_wet": "% of years wet"}))

write_geotiff(dyn_a, out_a / f"{name_a}_wetness_history.tif", DYNAMICS_COLORS)
class_table_for_display(classes_a).to_csv(out_a / f"{name_a}_wetness_history_classes.csv")
if len(patches_a):
    patches_a.to_file(out_a / f"{name_a}_wetland_patches.gpkg", layer="wetland_patches", driver="GPKG"); write_kmz(patches_a, out_a / f"{name_a}_wetland_patches.kmz")
build_run_manifest(parameters={"area": name_a, "nYear": YEARS_PER_PERIOD, "thresholdWet": MINIMUM_WET_YEARS_PERCENT, "thresholdPersis": PERSISTENT_YEARS_PERCENT, "smallest_patch_pixels": SMALLEST_PATCH_PIXELS,
                               "composite": "AWEIsh annual 90th percentile, " + ("all months" if months_a is None else f"months {months_a}"), "sensor": "LandsatAll incl. Landsat 7 after 2003", "years": [yrs_a[0], yrs_a[-1]]},
                   output_path=out_a / f"{name_a}_run_manifest.json")
print("files:", out_a)

**How to read it**

- **The sentences at the top** say how much wetland the satellite record shows, in how many patches, and how much of the national map it agrees with.
- **The map** colours each wetland pixel by its wetness history:
  - *Persistent*: wet in nearly every year. The stable core.
  - *Intermittent*: wet in some years and dry in others, with no clear direction. It swings.
  - *Intensifying*: wet more often in the recent period than in the early period. *Diminishing*: the opposite.
  - *New*: dry throughout the early period but wet in the recent period. *Lost*: wet in the early period but dry throughout the recent period.

  The early and recent periods are the first and last N years of the record. You choose N and the thresholds in the form.
- **The table and the bar chart by wetland type** compare the satellite record with the national map. For each type of wetland in the map, they show how much of its mapped area the satellite record also shows as wetland. A low percentage is not necessarily a mistake: seeps are wet at the surface only briefly, and a 30 m yearly record sees them poorly.
- **Satellite patches that are outside the national map** are places to check, not new wetlands. Look at them in imagery or in the field. **Open sea, large lakes and reservoirs also show as persistent wetland**: if the largest patch is one of those (in the example the large persistent areas along the southern edge of the study box are the sea), leave it out of your reading or cut your area to the land.
- Change the settings and run again to see how much the answer depends on them. Report the settings you used with any map.

Files are in `outputs/agulhas_plain/`: a GeoTIFF with the class colours inside, the wetland patches (GeoPackage, KMZ for Google Earth, CSV), the tables, the figures and a record of your settings.

## Part B. One wetland through 36 years

**The question.** For a wetland we already know (a saline pan, OBJECTID 3462), how have its wetness and its cover changed, and does rainfall explain it?

This is the report you can run for your own wetland (load it in the *wetland outline* cell above and set *AREA* to **my own wetland**): wet area per year, its trend, the link to rainfall, the wetness-history classes, and cover types through time. Part B has two steps.

1. **Look at the rainfall.** Choose the wetland and the years. The notebook shows the typical rainfall in each month at the site and suggests a rainy season: the five wettest months that fall inside one calendar year. Rainy seasons differ from place to place, so read the chart instead of assuming.
2. **Choose the months, then run.** You make two choices, after you have seen the rainfall. Each has a drop-down with three options: *all year*, *suggested rainy season* or *my own range of months*. If you choose your own range, pick its first and last month from the two drop-downs next to it (both months are included).
   - **Months for the wetness history**: the months of each year used for the yearly water image. The default is *all year*. In a dry place where water comes only in the rainy season, using only those months can show the wetland more clearly; a longer window catches brief water outside them.
   - **Months for the cover types**: the months used for the cover-type image. The default is the suggested rainy season.

   Then set the dynamics settings, the same four as in Part A.

**A season that crosses New Year** (for example November to March) cannot be treated as one season. The yearly images are built per calendar year, so January to March would be combined with the November and December of the same year, which belong to a different season. The notebook warns you if your range crosses New Year; choose a range inside one calendar year when you can.

The sample data that come with the notebook use all months for the wetness history and June to October for the cover types, and run as they are without an account. With Earth Engine on, other months are downloaded and used; without it the sample data are used and the notebook says so.

In [ ]:
#@title Part B, step 1. Choose the wetland and look at its rainfall { display-mode: "form" }
AREA = "example: pan 3462"  #@param ["example: pan 3462", "my own wetland"]
FIRST_YEAR_B = 1990  #@param {type:"integer"}
LAST_YEAR_B = 2025  #@param {type:"integer"}

import calendar
def plot_rain_months(clim, cover_months=None, dyn_months=None):
    # Typical rainfall by month. Orange bars: the months used for the cover types. A dark outline: the months used for the AWEIsh dynamics composite.
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.bar(clim.index, clim.values, color=["#e67e22" if (cover_months and m in cover_months) else "#1a5276" for m in clim.index],
           edgecolor=["k" if (dyn_months and m in dyn_months) else "none" for m in clim.index], linewidth=1.5)
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(list("JFMAMJJASOND")); ax.set_ylabel("mm per month")
    ax.set_title("Typical monthly rainfall" + (" (orange: cover-type months" + ("; outlined: dynamics months)" if dyn_months else ")") if cover_months else ""), fontsize=10)
    plt.show()

example_b = AREA.startswith("example")
if example_b:
    aoi_b, window_b, name_b, y0b, y1b = example_area("pan_polygon"), example_area("pan"), "pan3462", 1990, 2025
    rain_b = cached_rain("pan3462_era5_monthly", bounds_of(window_b, 0), 1990, 2025)
else:
    assert own_wetland is not None, "Run the 'Your wetland outline' cell first."
    aoi_b, window_b, name_b, y0b, y1b = own_wetland, own_wetland, WETLAND_NAME, FIRST_YEAR_B, LAST_YEAR_B
    rain_b = rain_monthly(bounds_of(aoi_b), y0b, y1b)
suggest_b, wrap_b, clim_b = suggest_months(rain_b)
plot_rain_months(clim_b)
print("Wettest months on average:", ", ".join(f"{calendar.month_abbr[m]} {v:.0f} mm" for m, v in clim_b.sort_values(ascending=False).head(5).items()))
print(f"Suggested rainy season, the five wettest months inside one calendar year: {suggest_b} ({', '.join(calendar.month_abbr[m] for m in suggest_b)})")
if wrap_b[0] > wrap_b[-1]:
    print(f"Note: the five wettest months run across New Year ({wrap_b}). A yearly composite cannot use them, so the suggestion above is the best inside one calendar year.")
print("Rainfall is ERA5-Land, in cells about 10 km wide: it describes the region around the wetland, not the wetland itself. Next: in step 2, choose the months with the drop-downs.")

In [ ]:
#@title Part B, step 2. Choose the months and run { display-mode: "form" }
DYNAMICS_MONTHS_MODE = "all year"  #@param ["all year", "suggested rainy season", "my own range of months"]
DYNAMICS_FIRST_MONTH = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
DYNAMICS_LAST_MONTH = "December"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
COVER_TYPE_MONTHS_MODE = "suggested rainy season"  #@param ["all year", "suggested rainy season", "my own range of months"]
COVER_TYPE_FIRST_MONTH = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
COVER_TYPE_LAST_MONTH = "December"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
MINIMUM_WET_YEARS_PERCENT = 25  #@param {type:"slider", min:5, max:60, step:5}
PERSISTENT_YEARS_PERCENT = 75  #@param {type:"slider", min:50, max:100, step:5}
YEARS_PER_PERIOD = 3  #@param {type:"slider", min:1, max:10, step:1}
# How to choose the months: pick one option in each *_MONTHS_MODE drop-down.
#   "all year" uses every month. "suggested rainy season" uses the months suggested in step 1.
#   "my own range of months" uses the range from the first to the last month in the two drop-downs next to it (both included). Those two are ignored for the other options.
# DYNAMICS_*: the months of each year used for the yearly water image (the 90th percentile of AWEIsh) behind the wetness history and the wet-area chart.
# COVER_TYPE_*: the months used for the cover-type image (the median of MNDWI, NDVI and NDTI).
# MINIMUM_WET_YEARS_PERCENT, PERSISTENT_YEARS_PERCENT, YEARS_PER_PERIOD: the same dynamics settings as in Part A.

EX_COVER = [6, 7, 8, 9, 10]                                          # the months the cached example was built with
dyn_months = choose_months(DYNAMICS_MONTHS_MODE, DYNAMICS_FIRST_MONTH, DYNAMICS_LAST_MONTH, suggest_b)          # None means all year
months_b = choose_months(COVER_TYPE_MONTHS_MODE, COVER_TYPE_FIRST_MONTH, COVER_TYPE_LAST_MONTH, suggest_b) or list(range(1, 13))
if example_b and COVER_TYPE_MONTHS_MODE == "suggested rainy season":
    months_b = EX_COVER
    print(f"For the sample data, 'suggested rainy season' means the months they were built with ({', '.join(MONTH_NAMES[m - 1] for m in EX_COVER)}). The suggestion in step 1 is {suggest_b}; to use it, choose it as 'my own range of months' (this needs Earth Engine).")
if example_b and not GEE_READY and (dyn_months is not None or months_b != EX_COVER):
    print("Earth Engine is off, so the sample data are used: the water image over all months of each year, and the cover types from the sample months. Choosing other months needs Earth Engine.")
    dyn_months, months_b = None, EX_COVER
plot_rain_months(clim_b, months_b, dyn_months)
print("Wetness history, yearly water image (90th percentile of AWEIsh):", "all months of each year" if dyn_months is None else f"months {dyn_months}", "| cover-type image (median):", f"months {months_b}")
for lab, mm in (("wetness-history", dyn_months), ("cover-type", months_b)):
    if mm and mm != list(range(min(mm), max(mm) + 1)):
        print(f"Note: the {lab} months cross New Year. Each yearly image takes them from one calendar year, so it mixes the end of one season with the start of the next. Prefer months inside one calendar year.")
use_cache_aw, use_cache_wct = example_b and dyn_months is None, example_b and months_b == EX_COVER
print("Data:", "AWEIsh from the cache" if use_cache_aw else "AWEIsh downloaded from Earth Engine", "|", "cover types from the cache" if use_cache_wct else "cover types downloaded from Earth Engine", "(downloads take a few minutes)" if not (use_cache_aw and use_cache_wct) else "")
w_geom = geojson_of(window_b)
aw_b = cached("pan3462_aweish_annual", lambda: annual_aweish(w_geom, 1990, 2025)) if use_cache_aw else annual_aweish(w_geom, y0b, y1b, months=dyn_months)
wct_b = cached("pan3462_wct_wetseason", lambda: wet_season_indices(w_geom, 1990, 2025, EX_COVER)) if use_cache_wct else wet_season_indices(w_geom, y0b, y1b, months_b)
out_b = OUT / name_b; out_b.mkdir(exist_ok=True)

w, s, e, n = aoi_b.total_bounds; pad = 0.0045
aw_b = aw_b.rio.clip_box(w - pad, s - pad, e + pad, n + pad); wct_b = wct_b.rio.clip_box(w - pad, s - pad, e + pad, n + pad)
mask_b = inside_mask(aoi_b, aw_b); px_b = pixel_area_m2(aw_b); mpx = int(mask_b.sum()); area_ha_b = mpx * px_b / 1e4
ok_b = aw_b.notnull().any("time")
check_dynamics_settings(aw_b.sizes["time"], YEARS_PER_PERIOD, MINIMUM_WET_YEARS_PERCENT, PERSISTENT_YEARS_PERCENT)
dyn_b = classify_dynamics(aw_b, nYear=YEARS_PER_PERIOD, thresholdWet=MINIMUM_WET_YEARS_PERCENT, thresholdPersis=PERSISTENT_YEARS_PERCENT, nan_policy="valid").where(ok_b)
wf_b = compute_wet_frequency(aw_b, nan_policy="valid").where(ok_b)

# wet area by year: class_area_timeseries on a wet/dry raster; years with little valid data are NaN, not zero
wetdry = (aw_b > 0).where(aw_b.notnull()).where(mask_b).astype("float32")
wet_ts = usable_years(class_area_timeseries(wetdry, class_labels={0: "dry", 1: "wet"}, pixel_area=px_b, area_unit="ha"), mpx, 1)
gaps_b = [y for y in range(int(wet_ts.index.min()), int(wet_ts.index.max()) + 1) if y not in set(wet_ts.dropna().index)]
ws = wet_ts.dropna()
sl = linear_trend(xr.DataArray(ws.values, dims="time", coords={"time": pd.to_datetime([f"{y}-07-01" for y in ws.index])}, name="wet_ha"))
k = min(5, max(len(ws) // 3, 1))
rain_y = None
if rain_b is not None and len(ws) >= 8:
    ep = rain_b.set_index("date")["precip_mm"]
    end_m = max(dyn_months) if dyn_months else 12                          # the 12 months of rain that end with the last month of the yearly image
    rain_name = f"rain in the 12 months up to the end of {MONTH_NAMES[end_m - 1]} (mm)"
    rain_y = pd.Series({y: ep[pd.Timestamp(y - 1, end_m, 1) + pd.offsets.MonthBegin(1): pd.Timestamp(y, end_m, 1)].sum() for y in ws.index})
    rho, pval = spearmanr(rain_y.values, ws.values)
lines = [f"{name_b}: mapped area {area_ha_b:,.1f} ha.",
         f"Wet area (pixels whose water index is above 0) averaged {ws.mean():,.1f} ha a year, from {ws.min():,.1f} ha in {ws.idxmin()} to {ws.max():,.1f} ha in {ws.idxmax()}.",
         f"The mean of the last {k} years ({ws.iloc[-k:].mean():,.1f} ha) is {abs(100 * (ws.iloc[-k:].mean() / ws.iloc[:k].mean() - 1)):.0f}% {'higher' if ws.iloc[-k:].mean() >= ws.iloc[:k].mean() else 'lower'} than the first {k} ({ws.iloc[:k].mean():,.1f} ha).",
         f"Trend: the wet area changed by about {float(sl['slope']) * 10:+.1f} ha per decade. The statistical test (p = {float(sl['p_value']):.2f}) " + ("says this is a consistent direction." if float(sl['p_value']) < 0.05 else "does not separate it from no trend. Read it as a direction over noisy yearly values, not a precise rate.")]
if rain_y is not None: lines.append(f"Wet area and {rain_name}: correlation {rho:+.2f} (1 means they rise and fall together, 0 means no link; p = {pval:.3f}, {len(ws)} years).")
if gaps_b: lines.append(f"Years with no usable data inside the outline, left out of the statistics: {', '.join(map(str, gaps_b))}.")
lines.append(f"Inside the outline, {float(summarize_dynamics(dyn_b.where(mask_b), pixel_area=px_b, area_unit='ha')['area_ha'].sel(class_code=slice(2, 10)).sum()):,.1f} ha is classed as wetland.")
lines.append("Built from: the 90th percentile of the water index (AWEIsh) over " + ("all months" if dyn_months is None else f"months {dyn_months}") + f" of each year; cover types from the median of months {months_b}.")
lines.append(f"Dynamics settings: wetland if wet in at least {MINIMUM_WET_YEARS_PERCENT}% of years; persistent if wet in at least {PERSISTENT_YEARS_PERCENT}%; early and recent periods of {YEARS_PER_PERIOD} years.")
print("\n".join("- " + l for l in lines))

classes_b = summarize_dynamics(dyn_b.where(mask_b), pixel_area=px_b, area_unit="ha").to_dataframe().drop(columns=["class_name"]).join(pd.Series(DYNAMICS_CLASSES, name="class"))
show(class_table_for_display(classes_b))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.2))
plot_dynamics(dyn_b, ax=axs[0], title="Wetness history (classes)", legend_loc="lower right"); wf_b.plot.imshow(ax=axs[1], vmin=0, vmax=100, cmap="Blues", cbar_kwargs={"label": "% of years wet"}); axs[1].set_title("How often wet (% of years)")
for ax in axs: aoi_b.boundary.plot(ax=ax, color="k", linewidth=1.2); ax.set_aspect(1 / np.cos(np.radians(float(aw_b.y.mean())))); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
fig.savefig(out_b / f"{name_b}_wetness_history_maps.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.8)); ax.bar(wet_ts.index, wet_ts.values, color="#1a5276")
for y in gaps_b: ax.axvspan(y - 0.5, y + 0.5, color="#e5e8ea", zorder=0)
ax.axhline(area_ha_b, color="#c0392b", ls="--", lw=1, label="mapped area"); ax.legend(loc="upper left", fontsize=8); ax.set_ylabel("wet area (ha)"); ax.set_title(f"{name_b}: wet area by year" + (" (grey = no usable data)" if gaps_b else ""))
if rain_y is not None:
    ax2 = ax.twinx(); ax2.spines["right"].set_visible(True); ax2.plot(rain_y.index, rain_y.values, color="#e67e22", marker="o", ms=3); ax2.set_ylabel(rain_name, color="#e67e22", fontsize=8)
fig.tight_layout(); fig.savefig(out_b / f"{name_b}_wet_area_by_year.png", dpi=150, bbox_inches="tight"); plt.show()
wet_ts.rename("wet_area_ha").to_csv(out_b / f"{name_b}_wet_area_by_year.csv")
rain_b.to_csv(out_b / f"{name_b}_rainfall_monthly_era5land.csv", index=False)          # monthly ERA5-Land rainfall (mm) used for this run
if rain_y is not None:
    pd.DataFrame({"wet_area_ha": wet_ts, rain_name: rain_y}).rename_axis("year").to_csv(out_b / f"{name_b}_wet_area_and_rain_by_year.csv")

ev = detect_wet_events(aw_b, threshold=0.0)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, v, lab in ((axs[0], "first_on_year", "first wet year"), (axs[1], "last_on_year", "last wet year"), (axs[2], "longest_on_streak", "longest run of wet years")):
    ev[v].where(ok_b).plot.imshow(ax=ax, cmap="viridis" if v != "longest_on_streak" else "magma_r", cbar_kwargs={"label": lab}); aoi_b.boundary.plot(ax=ax, color="k", linewidth=1); ax.set_title(lab)
    ax.set_aspect(1 / np.cos(np.radians(float(aw_b.y.mean())))); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude" if v == "first_on_year" else "")
fig.savefig(out_b / f"{name_b}_wet_events.png", dpi=150, bbox_inches="tight"); plt.show()

# cover types through time. Missing pixels are put back to NaN after classifying (older versions return class 0 for them).
valid_b = wct_b[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable")
wct_cls = classify_wct_ema(wct_b[["MNDWI", "NDVI", "NDTI"]])["wetland_cover_type"].where(valid_b)
pxw = pixel_area_m2(wct_b["MNDWI"]); maskw = inside_mask(aoi_b, wct_b["MNDWI"]); mpxw = int(maskw.sum())
wts = class_area_timeseries(wct_cls.where(maskw), class_labels=WCT_CLASSES, pixel_area=pxw, area_unit="ha")
area_w = pd.DataFrame({WCT_CLASSES[c]: usable_years(wts, mpxw, c) for c in WCT_CLASSES}); area_w.index.name = "year"
cols = [WCT_CLASSES[c] for c in range(1, 6)]
fig, ax = plt.subplots(figsize=(11, 3.8)); ax.stackplot(area_w.index, [area_w[c].fillna(0) for c in cols], labels=cols, colors=[WCT_COLORS[c] for c in range(1, 6)])
for y in area_w.index[area_w.isna().all(axis=1)]: ax.axvspan(y - 0.5, y + 0.5, color="#e5e8ea", zorder=0)
ax.legend(loc="upper left", fontsize=7, ncol=3); ax.set_ylabel("area inside the outline (ha)"); ax.set_title(f"{name_b}: cover types by year (rainy-season composite, months {months_b}; grey = no data)")
fig.tight_layout(); fig.savefig(out_b / f"{name_b}_cover_types_by_year.png", dpi=150, bbox_inches="tight"); plt.show()
area_w.round(2).to_csv(out_b / f"{name_b}_cover_types_by_year.csv")
av = area_w.dropna(how="all"); kk = min(5, max(len(av) // 3, 1))
chg = pd.DataFrame({f"first {kk} yrs (ha)": av.iloc[:kk].mean().round(1), f"last {kk} yrs (ha)": av.iloc[-kk:].mean().round(1)}); chg["change (ha)"] = chg.iloc[:, 1] - chg.iloc[:, 0]
print("Cover-type area, first versus last years:"); show(chg); chg.to_csv(out_b / f"{name_b}_cover_types_change.csv")
sel = sorted(set([int(av.index.min()), int(av.index.max()), int(ws.idxmin()), int(ws.idxmax())]))
fig, axs = plt.subplots(1, len(sel), figsize=(4.3 * len(sel), 4.4), squeeze=False)
for ax, y in zip(axs[0], sel):
    plot_wct(wct_cls.isel(time=int(np.argmax(pd.DatetimeIndex(wct_cls.time.values).year == y))), ax=ax, title=str(y), add_colorbar=(y == sel[-1])); aoi_b.boundary.plot(ax=ax, color="k", linewidth=1); ax.set_aspect(1 / np.cos(np.radians(float(wct_b.y.mean()))))
fig.savefig(out_b / f"{name_b}_cover_type_maps.png", dpi=150, bbox_inches="tight"); plt.show()

# per polygon, and files
aoi_b = aoi_b.reset_index(drop=True).assign(poly=lambda d: d.index)
per = summarize_by_polygons(wf_b, aoi_b[["poly", "geometry"]], polygon_id_col="poly", stats=("mean", "count")).rename(columns={"mean": "mean_wet_frequency_pct", "count": "pixels"})
per["area_ha"] = (per["pixels"] * px_b / 1e4).round(2); per.to_csv(out_b / f"{name_b}_per_polygon.csv", index=False)
if len(per) > 1: print("Statistics per polygon:"); show(per)
write_geotiff(dyn_b, out_b / f"{name_b}_wetness_history.tif", DYNAMICS_COLORS)
write_geotiff(wct_cls.isel(time=-1), out_b / f"{name_b}_cover_types_{int(pd.DatetimeIndex(wct_cls.time.values).year[-1])}.tif", WCT_COLORS)
class_table_for_display(classes_b).to_csv(out_b / f"{name_b}_wetness_history_classes.csv"); (out_b / f"{name_b}_report.md").write_text(f"# {name_b}\n\n" + "\n".join("- " + l for l in lines) + "\n")
build_run_manifest(parameters={"area": name_b, "nYear": 3, "thresholdWet": MINIMUM_WET_YEARS_PERCENT, "thresholdPersis": 75, "years": [y0b, y1b], "wet_season_months": months_b, "dynamics_months": dyn_months or "all year",
                               "dynamics_composite": "AWEIsh annual 90th percentile", "cover_type_composite": "MNDWI, NDVI, NDTI median of the rainy-season months", "rainfall": "ERA5-Land monthly"},
                   output_path=out_b / f"{name_b}_run_manifest.json")
print("files:", out_b)

**How to read it**

- **Wet area by year**, with rainfall on the same chart. If the bars follow the line, rainfall is the main driver. The correlation in the sentences says how closely: 1 means they rise and fall together, 0 means no link.
- **The trend** is a direction over noisy yearly values, not a precise rate. A wetland driven by rainfall can show a trend only because the record starts in a wet or dry run.
- **Grey years had no usable data.** Some years have too few clear satellite scenes. They are left out of the statistics, not counted as dry.
- **The wetness-history map** shows which parts are persistent, drying, newly wet and so on (see Part A). **The 'how often wet' map** shows the share of years each pixel was wet. **The three timing maps** show the first year each pixel was wet, the last year it was wet, and its longest run of wet years.
- **Cover types by year** show whether open water, vegetation or moist soil change with wetness. The maps show the first and last years and the driest and wettest.
- **Emergent / floating vegetation can be dry land.** The green-vegetation signal looks the same over a wet reed bed and over a crop or a patch of grass. When a pixel falls in this class, check that water is really there (open water or wet soil nearby, a photo, the water values in the table, a field visit) before you treat it as wetland.
- **The months you choose change the result.** Compare a run with all months and a run with the rainy season only, and report which you used (the report and the run manifest record them).
- **Rainfall is regional.** ERA5-Land cells are about 10 km wide, so the rainfall chart and the correlation describe the region around the wetland, not the wetland itself. They are good for timing the rainy season, less so for explaining one small wetland.

The mapped polygon is the best map available, not ground truth: its validation fields are empty.

## Part C. Cover types this week: the Langebaan Ramsar site

**The question.** What cover types does the satellite see at Langebaan right now, so you can compare with your own field visit?

Sentinel-2 passes about every five days. The notebook takes the scenes of the last days, drops the cloudy ones, builds a median composite, and classifies it into cover types. It lists the scenes with their **date and time in South African time**, so you can match them to your visit.

**Cautions.** Langebaan is **tidal**: the image shows the tide at the moment of the satellite pass, which may differ from the tide at your visit. The classes describe what the satellite signal looks like, not a verified habitat. In particular, **emergent / floating vegetation can be dry land with green plants** (grass, crops, shrubs): check that water is present before you treat that class as wetland. Your field notes are the check.

- With Earth Engine on, this is **live**. Without it you see a sample snapshot downloaded shortly before the workshop; the cell says which.
- Add your **field points** (a CSV with `lat` and `lon` columns, or a KMZ, KML or GeoPackage of points) to get the cover type and the index values at each point.

In [ ]:
#@title Part C. Cover types this week { display-mode: "form" }
AREA = "example: Langebaan Ramsar site"  #@param ["example: Langebaan Ramsar site", "my own wetland"]
DAYS_BACK = 14  #@param {type:"slider", min:5, max:30, step:1}
MINIMUM_VALID_PERCENT = 40  #@param {type:"slider", min:10, max:90, step:5}
FIELD_POINTS = "none"  #@param ["none", "upload now", "path in Drive or Colab"]
FIELD_POINTS_PATH = ""  #@param {type:"string"}

if AREA.startswith("example"):
    aoi_c, name_c = example_area("langebaan"), "langebaan"
else:
    assert own_wetland is not None, "Run the 'Your wetland outline' cell first."
    aoi_c, name_c = own_wetland, WETLAND_NAME
out_c = OUT / (name_c + "_now"); out_c.mkdir(exist_ok=True)
end = dt.date.today() + dt.timedelta(days=1); start = end - dt.timedelta(days=DAYS_BACK + 1)
if GEE_READY:
    ds_c = fetch(aoi=geojson_of(aoi_c), start=start.isoformat(), end=end.isoformat(), sensor="Sentinel2", index=["MNDWI", "NDVI", "NDTI"], scale=20, temporal_aggregation="all", max_cloud_cover=60.0, project=GEE_PROJECT or None)
    ds_c = xr.Dataset({v: fin(ds_c[v]) for v in ("MNDWI", "NDVI", "NDTI")}); source_c = f"live from Earth Engine, {dt.date.today()}"
elif AREA.startswith("example") and (CACHE / "langebaan_s2_recent.nc").exists():
    ds_c = cached("langebaan_s2_recent", None); meta = json.loads((CACHE / "langebaan_recent_meta.json").read_text()) if (CACHE / "langebaan_recent_meta.json").exists() else {}
    source_c = f"sample snapshot downloaded {meta.get('downloaded') or meta.get('exported', 'earlier')}; switch on Earth Engine for a live map"
else:
    need_gee("A recent map of your own area")
print("Data:", source_c)

mask_c = inside_mask(aoi_c, ds_c["MNDWI"]); t = pd.DatetimeIndex(ds_c.time.values)
valid_all = ds_c[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable") & mask_c
scenes = pd.DataFrame({"time_utc": t, "time_local": t.tz_localize("UTC").tz_convert("Africa/Johannesburg").tz_localize(None), "valid_pct": (100 * valid_all.sum(("x", "y")) / int(mask_c.sum())).values.round(1)})
scenes["used"] = scenes["valid_pct"] >= MINIMUM_VALID_PERCENT
if not scenes["used"].any(): raise ValueError(f"No scene has at least {MINIMUM_VALID_PERCENT}% valid pixels inside the outline. Increase DAYS_BACK, lower the minimum, or try after the next clear satellite pass.")
sel_c = ds_c.isel(time=np.where(scenes["used"].values)[0])
comp = xr.Dataset({v: sel_c[v].median("time", skipna=True) for v in ("MNDWI", "NDVI", "NDTI")}).rio.write_crs("EPSG:4326")
vc = comp[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable") & mask_c
wct_now = classify_wct_ema(comp)["wetland_cover_type"].where(vc)
px_c = pixel_area_m2(comp["MNDWI"])
now_tab = summarize_wct(wct_now.where(mask_c), pixel_area=px_c, area_unit="ha").to_dataframe()[["class_name", "pixel_count", "area_ha"]]
f0, f1 = scenes.loc[scenes["used"], "time_local"].min(), scenes.loc[scenes["used"], "time_local"].max()
print(f"{int(scenes['used'].sum())} of {len(scenes)} scenes used (at least {MINIMUM_VALID_PERCENT}% valid), from {f0:%d %b %Y %H:%M} to {f1:%d %b %Y %H:%M} South African time. {100 * float(vc.sum() / mask_c.sum()):.0f}% of the outline has a valid composite.")
show(scenes.rename(columns={"time_utc": "Time (UTC)", "time_local": "Time (South Africa)", "valid_pct": "% of the outline with clear data", "used": "Used in the map"}))
show(now_tab.rename(columns={"class_name": "Cover type", "pixel_count": "Pixels", "area_ha": "Area (ha)"}))
print("Note: 'Emergent / Floating Vegetation' can also be dry land with green plants. Check that water is present before treating it as wetland.")

# context: the usual rainy-season cover types, 2016 to 2025 (Landsat, 30 m, a different sensor and season: context only)
if AREA.startswith("example"):
    usual_ds = cached("langebaan_wct_wetseason", lambda: mask_to(wet_season_indices(geojson_of(langebaan_box()), 1990, 2025, [6, 7, 8, 9, 10]), aoi_c))
    uv = usual_ds[["MNDWI", "NDVI", "NDTI"]].to_array().notnull().all("variable")
    ucls = classify_wct_ema(usual_ds[["MNDWI", "NDVI", "NDTI"]])["wetland_cover_type"].where(uv).isel(time=slice(-10, None))
    um = inside_mask(aoi_c, usual_ds["MNDWI"]); uts = class_area_timeseries(ucls.where(um), class_labels=WCT_CLASSES, pixel_area=pixel_area_m2(usual_ds["MNDWI"]), area_unit="ha")
    usual = pd.Series({WCT_CLASSES[c]: usable_years(uts, int(um.sum()), c).mean() for c in WCT_CLASSES})
    cmp_ = pd.DataFrame({"now (ha)": now_tab["area_ha"].values, "usual rainy season, 2016 to 2025 (ha)": usual.values.round(1)}, index=now_tab["class_name"].values); cmp_["difference (ha)"] = (cmp_.iloc[:, 0] - cmp_.iloc[:, 1]).round(1)
    print("\nCompared with the usual rainy season. Different sensor, season and resolution: use as context only."); show(cmp_); cmp_.to_csv(out_c / f"{name_c}_now_vs_usual.csv")

fig, ax = plot_wct(wct_now, title=f"{name_c}: cover types, {f0:%d %b} to {f1:%d %b %Y}", figsize=(8, 7)); aoi_c.boundary.plot(ax=ax, color="k", linewidth=0.5); ax.set_aspect(1 / np.cos(np.radians(float(wct_now.y.mean()))))
pts_path = upload_file() if FIELD_POINTS == "upload now" else (FIELD_POINTS_PATH if FIELD_POINTS == "path in Drive or Colab" else None)
if pts_path:
    pts = read_points(pts_path); rows = []
    for r in pts.itertuples():
        iy, ix = int(np.abs(wct_now.y.values - r.geometry.y).argmin()), int(np.abs(wct_now.x.values - r.geometry.x).argmin()); c = wct_now.values[iy, ix]
        rows.append({"point_id": r.point_id, "lon": round(r.geometry.x, 5), "lat": round(r.geometry.y, 5), "cover type now": WCT_CLASSES.get(int(c), "") if np.isfinite(c) else "no valid data",
                     **{v: round(float(comp[v].values[iy, ix]), 2) for v in ("MNDWI", "NDVI", "NDTI")}})
        ax.plot(r.geometry.x, r.geometry.y, "k^", ms=7, mfc="none")
    ptab = pd.DataFrame(rows); print("\nValues at your field points (write down the tide at your visit):"); show(ptab); ptab.to_csv(out_c / f"{name_c}_field_points_now.csv", index=False)
fig.savefig(out_c / f"{name_c}_cover_types_now.png", dpi=150, bbox_inches="tight"); plt.show()
scenes.to_csv(out_c / f"{name_c}_scenes.csv", index=False); now_tab.to_csv(out_c / f"{name_c}_cover_types_now.csv")
write_geotiff(wct_now, out_c / f"{name_c}_cover_types_now.tif", WCT_COLORS)
build_run_manifest(parameters={"area": name_c, "sensor": "Sentinel-2 SR harmonized", "scale_m": 20, "window": f"{start} to {end}", "min_valid_percent": MINIMUM_VALID_PERCENT, "composite": "median of used scenes", "data": source_c},
                   output_path=out_c / f"{name_c}_now_run_manifest.json")
print("files:", out_c)

**How to read it**

- **The scene list** shows which images went into the map and when they were taken (South African time). A scene with few valid pixels is mostly cloud and is left out.
- **The map and class table** give the cover types now. **The comparison with the usual rainy season** is context only: Landsat composites from June to October, 2016 to 2025, a different sensor, season and resolution.
- **Field points:** the table gives the class and the three index values (MNDWI for open water, NDVI for green vegetation, NDTI for turbidity). If your notes and the class disagree, the index values tell you which way the pixel leans.
- **Emergent / floating vegetation can be dry land.** Green plants look the same over a wet reed bed and over grass or a crop. Check that water is present at your point (a photo, wet soil, the water value in the table) before you call it wetland.
- If the map looks wrong, check the scene dates first: haze, cloud or a different tide is the usual reason.

## Keep your results

Everything is written to the `outputs` folder, which Colab deletes when the session ends. Download a zip, or copy it to a Google Drive folder.

In [ ]:
#@title Download or save your results { display-mode: "form" }
SAVE_TO = "zip in the outputs folder"  #@param ["zip in the outputs folder", "download zip (Colab)", "copy to Google Drive (Colab)", "copy to a folder"]
FOLDER = ""  #@param {type:"string"}
# Google Drive: a folder name inside My Drive (default WetlandMapper_results). Copy to a folder: any folder path on your computer.

DRIVE_ROOT = Path("/content/drive/MyDrive")
zip_path = Path(shutil.make_archive(str(ROOT / "wetlandmapper_results"), "zip", OUT))     # the zip sits next to outputs/, not inside it
print("Zipped results:", zip_path)

def copy_results(dest):
    dest = Path(dest).expanduser(); dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT, dest / "outputs", dirs_exist_ok=True); shutil.copy(zip_path, dest / zip_path.name)
    print("Copied the outputs folder and the zip to", dest.resolve())

if SAVE_TO == "zip in the outputs folder":
    print("The files stay in", OUT.resolve(), "(on your own computer they are kept there; Colab deletes them when the session ends).")
elif SAVE_TO == "download zip (Colab)":
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
    else:
        print("Not in Colab, so there is nothing to download: the zip is at", zip_path.resolve())
elif SAVE_TO == "copy to Google Drive (Colab)":
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        copy_results(DRIVE_ROOT / (FOLDER or "WetlandMapper_results"))
    else:
        print("Not in Colab. Choose 'copy to a folder' and give a folder path, or use the zip at", zip_path.resolve())
else:
    if not FOLDER:
        raise ValueError("Give a folder path in FOLDER, for example ~/wetland_results")
    copy_results(FOLDER)

Open the GeoTIFFs in QGIS: the class colours are inside the file. The wetland patches come as a GeoPackage (QGIS, ArcGIS) and a KMZ (Google Earth, a phone). `*_report.md` holds the sentences; `*_run_manifest.json` records your settings and the software version: keep it with the maps.

## If something goes wrong

| Message | What to do |
|---|---|
| "needs Earth Engine" | Tick *USE_EARTH_ENGINE* in the setup cell, enter your project ID, run setup again |
| "sample data ... not in data/cache/ yet" | The workshop leader has not downloaded them yet, or switch on Earth Engine to download them once |
| "No polygons found" | Your file holds points or lines. Draw a polygon and save again |
| "has no ... coordinate system" | A shapefile is missing its `.prj`. Upload it, or save as GeoPackage |
| "too large for this notebook" | Use a smaller outline |
| "No scene has at least 40% valid pixels" | Increase DAYS_BACK, lower the minimum, or try after the next clear satellite pass |
| The run is slow | Earth Engine fetches 36 years one by one: minutes for a small wetland, much longer for a large area |
| A year is missing | Earth Engine returned no scenes or a download failed twice. It is left out of the statistics |

## Reading results, and what this notebook does not do

- Open water is seen well. **Small, vegetated or briefly wet features** (springs, seeps, reed beds) are seen poorly: a 30 m pixel covers 0.09 ha.
- A yearly image records the wettest cloud-free moment of the months you choose, not the whole year. **Cover-type classes describe the satellite signal**, not a verified habitat; in tidal places the tide at the satellite pass matters.
- **Emergent / floating vegetation can be dry land.** The vegetation signal is the same over a wet reed bed and over grass or a crop. Check that water is present before you treat that class as wetland.
- A wetland on **steep, shaded ground** can be confused with shadow, and in a **dry climate** water can be too brief for a yearly composite. WetlandMapper has options for both, which this notebook does not run: `fetch(..., dem_mask=True)` masks steep ground, and `fetch(..., climate_adaptive=True)` chooses the month to sample from rainfall. See the WetlandMapper documentation.
- **Always report with the map:** the water index, the satellite, how each yearly image was built (percentile and months), the dynamics settings (wet percentage, persistent percentage, years per period), the years and the software version. The run manifest has them.

### Data and citation

- Software: Singh, M. (2026). WetlandMapper. https://doi.org/10.5281/zenodo.18967176
- Dynamics: Singh, M., and Sinha, R. (2022). *Remote Sensing Letters*, 13(1), 1-13. https://doi.org/10.1080/2150704X.2021.1980919
- Cover types: Singh, M., Allaka, S., Gupta, P. K., Patel, J. G., and Sinha, R. (2022). *Environmental Monitoring and Assessment*, 194(12), 878. https://doi.org/10.1007/s10661-022-10541-7
- Agulhas Plain Ramsar wetlands layer, supplied by SANParks (Ruth-Mary Fisher); a subset of the National Wetland Map 5. Please cite: Van Deventer, H., Smith-Adao, L., Mbona, N., Petersen, C., Skowno, A., Collins, N.B., Grenfell, M., Job, N., Lötter, M., Ollis, D., Scherman, P., Sieben, E., and Snaddon, K. (2018). South African National Biodiversity Assessment 2018: Technical Report, Volume 2a: South African Inventory of Inland Aquatic Ecosystems (SAIIAE), version 3. CSIR and SANBI, Pretoria. http://hdl.handle.net/20.500.12143/5847
- Langebaan Ramsar site boundary: Ramsar Sites Information Service (https://rsis.ramsar.org).
- Landsat Collection 2 Level-2 (USGS and NASA), Sentinel-2 surface reflectance (ESA and Copernicus), ERA5-Land (Copernicus and ECMWF), through Google Earth Engine (Gorelick et al., 2017, *Remote Sensing of Environment* 202, 18-27).